<a href="https://colab.research.google.com/github/bernardo-maltez/DbManager/blob/main/projects/ECoG/load_ECoG_faceshouses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Data retrieval
import os, requests

fname = 'faceshouses.npz'
url = "https://osf.io/argh7/download"

if not os.path.isfile(fname):
  try:
    r = requests.get(url)
  except requests.ConnectionError:
    print("!!! Failed to download data !!!")
  else:
    if r.status_code != requests.codes.ok:
      print("!!! Failed to download data !!!")
    else:
      with open(fname, "wb") as fid:
        fid.write(r.content)

In [12]:
# @title Imports

from matplotlib import rcParams
from matplotlib import pyplot as plt
rcParams['figure.figsize'] = [20, 4]
rcParams['font.size'] = 15
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['figure.autolayout'] = True

In [3]:
# @title Data loading
import numpy as np

alldat = np.load(fname, allow_pickle=True)['dat']

dict_keys(['t_off', 'stim_id', 't_on', 'srate', 'V', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])
dict_keys(['stim_id', 'stim_cat', 'stim_noise', 't_on', 't_off', 'key_press', 'V', 'categories', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])


In [17]:
# @title Configure

subject_index = 1 #1 and 2 missing data on dat2

dat1 = alldat[subject_index][0]
dat2 = alldat[subject_index][1]

print(dat1.keys())
print(dat2.keys())

dict_keys(['t_off', 'stim_id', 't_on', 'srate', 'V', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])
dict_keys(['stim_id', 'stim_cat', 'stim_noise', 't_on', 't_off', 'key_press', 'V', 'categories', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])


In [23]:
# @title Get broadband
from scipy import signal

V = dat1['V'].astype('float32')

# Remove slower
b, a = signal.butter(3, [50], btype='high', fs=1000)
V = signal.filtfilt(b, a, V, 0)

# power
V = np.abs(V)**2

# Remove fast variations
b, a = signal.butter(3, [10], btype='low', fs=1000)
V = signal.filtfilt(b, a, V, 0)

# Normalize to 0
V = V/V.mean(0)

In [57]:
# 1. Set the time window
# 2. Get avarage
# 3. Save avarage per electrode (feature) on trial (Matrix M)
# 2. 100x1 matrix with answers

# 1. Define your parameters
chunk_size = 800
stim_V = V[dat1['t_on'][0]:]  # Your starting slice
num_channels = stim_V.shape[1]

# 2. Truncate the array so it's perfectly divisible by 400
# (removes any leftover samples at the very end in one step)
num_chunks = len(stim_V) // chunk_size
truncated_length = num_chunks * chunk_size
stim_V_truncated = stim_V[:truncated_length]

# 3. Reshape into a 3D array: (Chunks, Timepoints per chunk, Channels)
# This operation is instantaneous because it creates a "view" without copying memory
stim_V_3d = stim_V_truncated.reshape(num_chunks, chunk_size, num_channels)

# 4. Calculate the mean over the time axis (axis=1) for all chunks simultaneously
# Output shape: (num_chunks, num_channels)
stim_values = np.mean(stim_V_3d, axis=1)